# Кейс 3. Шаг 5: клиентские логи (H2 трафик, H4 ошибки, H10 версии) и исправления по БД

1. **Исправления по БД**: три запроса из ноутбука 03 (служебное слово и приоритет `NOT`), H7 по правильному признаку автопродления (`subscribe.pre_payment`; в `orders` этот флаг стоит только у автосписаний продления), точность момента покупки после триала, судьба зависших продлений мая–июня, реклама у гостей (исправлен учёт пустых значений).
2. **Логи**: скачивается `logs.zip` (4,7 ГБ, на диск Colab, не на Drive), внешний архив распаковывается, вложенные читаются из zip без распаковки. Разбор идёт регулярными выражениями по байтам, только нужные строки.
   - **H2**: счётчики `TrafficLimitBloc` (лимит и потребление в байтах) → реальное потребление на бесплатном тарифе и доля упирающихся в лимит.
   - **H4**: типы ошибок на инцидент, их сочетания и события перед первой ошибкой.
   - **H10**: ошибки по версиям приложения и ОС.

Итог: `h_logs_results.txt` и графики в `case3/fig/`. Ориентировочное время: скачивание 5–10 минут, разбор логов 20–40 минут.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/case3"
WORK = "/content/case3"
PUBLIC_URL = "https://disk.yandex.ru/d/UYuFNCrHvWR3DQ"
import os, re, json, time, zipfile, traceback, collections, urllib.parse, urllib.request
import numpy as np, pandas as pd, duckdb, matplotlib.pyplot as plt
from math import sqrt
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from google.colab import drive
drive.mount("/content/drive")
PARQUET, FIG = f"{DRIVE_DIR}/parquet", f"{DRIVE_DIR}/fig"
os.makedirs(FIG, exist_ok=True); os.makedirs(WORK, exist_ok=True)
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
for f in sorted(os.listdir(PARQUET)):
    if f.endswith(".parquet") and not f.startswith("user_connection_logs_"):
        con.execute(f"CREATE OR REPLACE VIEW {f[:-8]} AS SELECT * FROM '{PARQUET}/{f}'")
con.execute("CREATE OR REPLACE VIEW mart AS SELECT * FROM mart_users")
con.execute("""CREATE OR REPLACE TABLE ord AS
    SELECT *, try_cast(amount AS DOUBLE) AS amt,
           CASE WHEN trial = 't' THEN 'trial'
                WHEN coalesce(sum(CASE WHEN trial = 'f' AND status = 'paid' THEN 1 ELSE 0 END)
                     OVER (PARTITION BY user_id ORDER BY created_at ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) = 0
                THEN 'first' ELSE 'repeat' END AS order_type
    FROM orders""")
DATA_END = "2026-09-29 00:00:00"

REPORT = []
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
def out(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)
def show(title, sql, n=60):
    out(f"\n## {title}")
    try:
        df = con.sql(sql).df(); out(df.head(n).to_string(index=False) if len(df) else "(пусто)"); return df
    except Exception as e:
        out(f"ОШИБКА: {e}")
def save():
    open(f"{DRIVE_DIR}/h_logs_results.txt", "w", encoding="utf-8").write("\n".join(REPORT))
def section(name, fn):
    out(f"\n{'=' * 70}\n# {name}")
    try:
        fn()
    except Exception:
        out("ОШИБКА РАЗДЕЛА:\n" + traceback.format_exc(limit=3))
    save()
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(float(c - h), 4), round(float(c + h), 4))

C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "violet": "#6250d6",
     "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df"}
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink"],
    "xtick.color": C["ink2"], "ytick.color": C["ink2"], "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "lines.linewidth": 2})
pct = plt.FuncFormatter(lambda v, _: f"{v:.0%}")

## 1. Исправления по БД

In [ ]:
def db_fixes():
    show("H5: причины отказа Yookassa по типу заказа", """
        SELECT o.order_type, t.yk_cancel_reason, count(*) n,
               round(count(*) / sum(count(*)) OVER (PARTITION BY o.order_type), 4) share_n
        FROM transactions t JOIN ord o ON o.id = t.order_id
        WHERE t.yk_cancel_reason IS NOT NULL GROUP BY 1, 2
        QUALIFY row_number() OVER (PARTITION BY o.order_type ORDER BY count(*) DESC) <= 5
        ORDER BY 1, n DESC""")
    show("H5: что происходит после неуспешного заказа (заказы старше 7 дней)", f"""
        SELECT o.order_type, count(*) failed_orders,
               round(avg(CAST(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 1 HOUR) AS INT)), 4) paid_within_1h,
               round(avg(CAST(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 7 DAY) AS INT)), 4) paid_within_7d,
               round(avg(CAST(NOT EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at) AS INT)), 4) never_paid_after
        FROM ord o WHERE o.status = 'failed' AND o.created_at < TIMESTAMP '{DATA_END}' - INTERVAL 7 DAY
        GROUP BY 1 ORDER BY 1""")
    show("H6: причины отказа Yookassa, длинный период vs месяц", """
        SELECT CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END AS period_kind, t.yk_cancel_reason, count(*) n,
               round(count(*) / sum(count(*)) OVER (PARTITION BY CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END), 4) share_n
        FROM transactions t JOIN ord o ON o.id = t.order_id
        WHERE t.yk_cancel_reason IS NOT NULL AND o.order_type <> 'trial' AND o.period_id IS NOT NULL
        GROUP BY 1, 2
        QUALIFY row_number() OVER (PARTITION BY CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END ORDER BY count(*) DESC) <= 4
        ORDER BY 1, n DESC""")
    show("H6: доля выручки длинных периодов (Yookassa)", """
        SELECT CASE WHEN period_id = '1' THEN 'месяц' ELSE 'длинный' END AS period_kind, round(sum(amt)) revenue_rub,
               round(sum(amt) / sum(sum(amt)) OVER (), 4) share_revenue
        FROM ord WHERE order_type <> 'trial' AND status = 'paid' AND provider_id = '2' AND period_id IS NOT NULL GROUP BY 1""")

    out("\n## Что означает pre_payment")
    show("orders.pre_payment по типу заказа", "SELECT order_type, pre_payment, status, count(*) n FROM ord GROUP BY ALL ORDER BY 1, 2, 3")

    con.execute(f"""CREATE OR REPLACE TABLE h7_base AS
        WITH fp AS (SELECT o.id, o.user_id, o.created_at, o.product_id, s.pre_payment AS sub_autorenew, s.expired_time, s.disabled_at
                    FROM ord o JOIN subscribe s ON s.order_id = o.id
                    WHERE o.order_type = 'first' AND o.status = 'paid' AND o.period_id = '1')
        SELECT fp.*, EXISTS (SELECT 1 FROM ord p WHERE p.user_id = fp.user_id AND p.status = 'paid' AND p.trial = 'f'
                             AND p.created_at > fp.created_at + INTERVAL 1 DAY
                             AND p.created_at <= fp.expired_time + INTERVAL 7 DAY) AS renewed,
               EXISTS (SELECT 1 FROM ord p WHERE p.user_id = fp.user_id AND p.status = 'paid' AND p.pre_payment = 't'
                             AND p.created_at > fp.created_at + INTERVAL 1 DAY
                             AND p.created_at <= fp.expired_time + INTERVAL 7 DAY) AS renewed_auto
        FROM fp WHERE fp.expired_time + INTERVAL 7 DAY <= TIMESTAMP '{DATA_END}'""")
    d = show("H7: продление первой месячной подписки по признаку автопродления подписки", """
        SELECT sub_autorenew, count(*) n, sum(renewed::INT) renewed, round(avg(renewed::INT), 4) renew_rate,
               round(avg(renewed_auto::INT), 4) renewed_by_autocharge,
               round(avg(CAST(disabled_at IS NOT NULL AND disabled_at < expired_time AS INT)), 4) disabled_before_expiry
        FROM h7_base GROUP BY 1 ORDER BY 1""")
    if d is not None and len(d) >= 2 and {"t", "f"} <= set(d["sub_autorenew"]):
        a, b = d.set_index("sub_autorenew").loc["t"], d.set_index("sub_autorenew").loc["f"]
        out("95% ДИ продления: с автопродлением", wilson(a["renewed"], a["n"]), "; без", wilson(b["renewed"], b["n"]))
    show("H7: продление по пути к покупке и автопродлению", """
        SELECT CASE WHEN m.first_trial_paid < h.created_at THEN 'после триала' ELSE 'без триала' END AS path,
               h.sub_autorenew, count(*) n, round(avg(h.renewed::INT), 4) renew_rate
        FROM h7_base h JOIN mart m USING (user_id) GROUP BY ALL ORDER BY 1, 2""")
    show("H7: удержание: доля продливших k раз подряд (первые покупки на месяц до 2026-04-01)", """
        WITH fp AS (SELECT user_id, created_at FROM ord WHERE order_type = 'first' AND status = 'paid' AND period_id = '1'
                    AND created_at < '2026-04-01'),
             k AS (SELECT fp.user_id, count(p.id) n_paid_after
                   FROM fp LEFT JOIN ord p ON p.user_id = fp.user_id AND p.status = 'paid' AND p.trial = 'f'
                        AND p.created_at > fp.created_at + INTERVAL 1 DAY GROUP BY 1)
        SELECT round(avg((n_paid_after >= 1)::INT), 4) ge1, round(avg((n_paid_after >= 2)::INT), 4) ge2,
               round(avg((n_paid_after >= 3)::INT), 4) ge3, round(avg((n_paid_after >= 5)::INT), 4) ge5, count(*) users
        FROM k""")

    show("H3: покупка после триала — насколько точно она совпадает с концом триала", """
        WITH t AS (SELECT o.user_id, s.expired_time trial_end FROM ord o JOIN subscribe s ON s.order_id = o.id
                   WHERE o.order_type = 'trial' AND o.status = 'paid'),
             x AS (SELECT m.user_id, epoch(m.first_regular_paid - t.trial_end) / 60 AS diff_min
                   FROM mart m JOIN t ON t.user_id = m.user_id
                   WHERE m.first_regular_paid > m.first_trial_paid
                     AND m.first_regular_paid < m.first_trial_paid + INTERVAL 30 DAY)
        SELECT CASE WHEN diff_min < -60 THEN 'раньше конца триала > 1 ч'
                    WHEN diff_min < -5 THEN 'за 5–60 мин до конца'
                    WHEN diff_min <= 5 THEN '±5 мин от конца'
                    WHEN diff_min <= 60 THEN 'через 5–60 мин'
                    WHEN diff_min <= 24 * 60 THEN 'через 1–24 ч'
                    ELSE 'позже суток' END AS bucket,
               count(*) n, round(count(*) / sum(count(*)) OVER (), 4) share_n
        FROM x GROUP BY 1 ORDER BY min(diff_min)""")

    show("H5: зависшие продления 25.05–28.06.2026 — чем закончились", f"""
        WITH p AS (SELECT * FROM ord WHERE order_type = 'repeat' AND status = 'pending'
                   AND created_at >= '2026-05-25' AND created_at < '2026-06-29')
        SELECT count(*) pending_orders, count(DISTINCT user_id) users, round(sum(amt)) amount_rub,
               round(avg(CAST(EXISTS (SELECT 1 FROM ord q WHERE q.user_id = p.user_id AND q.status = 'paid'
                     AND q.created_at > p.created_at AND q.created_at <= p.created_at + INTERVAL 14 DAY) AS INT)), 4) paid_within_14d,
               round(avg(CAST(NOT EXISTS (SELECT 1 FROM ord q WHERE q.user_id = p.user_id AND q.status = 'paid'
                     AND q.created_at > p.created_at) AS INT)), 4) never_paid_after
        FROM p""")
    show("H5: то же для сравнения — обычные продления 2026-04 (не в сбое)", """
        WITH p AS (SELECT * FROM ord WHERE order_type = 'repeat' AND created_at >= '2026-04-01' AND created_at < '2026-05-01')
        SELECT status, count(*) n, round(avg(CAST(NOT EXISTS (SELECT 1 FROM ord q WHERE q.user_id = p.user_id AND q.status = 'paid'
                     AND q.created_at > p.created_at) AS INT)), 4) never_paid_after
        FROM p GROUP BY 1""")

    show("H2/H8: реклама у гостей (исправлен учёт пустых значений)", """
        SELECT CASE WHEN coalesce(free_conn_days, 0) = 0 THEN '0' WHEN free_conn_days = 1 THEN '1'
                    WHEN free_conn_days <= 7 THEN '2–7' WHEN free_conn_days <= 30 THEN '8–30' ELSE '> 30' END AS free_days,
               count(*) guests, round(avg(CAST(coalesce(n_adv, 0) > 0 AS INT)), 4) watched_ads,
               round(avg(coalesce(n_adv, 0)), 1) avg_views_all, round(median(n_adv), 1) median_views_watchers
        FROM mart WHERE role = 'guest' GROUP BY 1 ORDER BY min(coalesce(free_conn_days, 0))""")
    show("H8: есть ли у гостей устройства в devices", """
        SELECT role, count(*) users, round(avg(CAST(n_devices > 0 AS INT)), 4) with_devices FROM mart GROUP BY 1""")
section("Исправления по БД", db_fixes)

## 2. Скачивание логов

In [ ]:
API = "https://cloud-api.yandex.net/v1/disk/public/resources"
def api_get(url, retries=5):
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r: return json.load(r)
        except Exception as e:
            print(f"  API: {e}, повтор через {2 ** i} с"); time.sleep(2 ** i)
    raise RuntimeError("Яндекс.Диск недоступен")
def walk(path="/"):
    offset = 0
    while True:
        q = urllib.parse.urlencode({"public_key": PUBLIC_URL, "path": path, "limit": 1000, "offset": offset})
        page = api_get(f"{API}?{q}")["_embedded"]
        for it in page["items"]:
            if it["type"] == "dir": yield from walk(it["path"])
            else: yield it
        offset += len(page["items"])
        if offset >= page["total"] or not page["items"]: return

LOGS_ZIP, LOGS_DIR = f"{WORK}/logs.zip", f"{WORK}/logs"
if not os.path.exists(f"{LOGS_DIR}/_done"):
    item = next(it for it in walk("/") if it["name"].lower() == "logs.zip")
    if not (os.path.exists(LOGS_ZIP) and os.path.getsize(LOGS_ZIP) == item["size"]):
        q = urllib.parse.urlencode({"public_key": PUBLIC_URL, "path": item["path"]})
        t0 = time.time()
        with urllib.request.urlopen(api_get(f"{API}/download?{q}")["href"], timeout=600) as r, open(LOGS_ZIP, "wb") as f:
            while chunk := r.read(8 << 20): f.write(chunk)
        print(f"logs.zip скачан за {time.time() - t0:.0f} с")
    with zipfile.ZipFile(LOGS_ZIP) as z: z.extractall(LOGS_DIR)
    open(f"{LOGS_DIR}/_done", "w").close()
LOG_ROOT = next(cur for cur, ds, fs in os.walk(LOGS_DIR) if sum(d.isdigit() for d in ds) > 10)
ZIPS = [(d, os.path.join(LOG_ROOT, d, f)) for d in sorted(os.listdir(LOG_ROOT)) if d.isdigit()
        for f in os.listdir(os.path.join(LOG_ROOT, d)) if f.lower().endswith(".zip")]
print("архивов:", len(ZIPS))

## 3. Разбор архивов

Для каждого архива (инцидента) извлекается:
- заголовок приложения: платформа, версия ОС, версия приложения;
- счётчики трафика `TrafficLimitBloc` (дата, лимит и потребление в байтах; флаг `limitReached`);
- число строк ошибок по типам в журнале приложения и в журнале ошибок VPN-ядра;
- последние события перед первой ошибкой в каждом журнале приложения.

IP, почта и токены в результат не попадают: извлекаются только числа, даты, версии и шаблоны событий с цифрами, заменёнными на 9.

In [ ]:
HDR = re.compile(rb"Running on Os\.(\w+); OS version ([^;\n]+); App version ([\w.\-+]+)")
TRAF = re.compile(rb"(\d{4}-\d{2}-\d{2})T[\d:.]+\s+\[\w\]\s+\[TrafficLimitBloc\] Traffic info changed: (\d+) bytes limit, (\d+) bytes used")
UPD = re.compile(rb"(\d{4}-\d{2}-\d{2})T[\d:.]+\s+\[\w\]\s+\[TrafficLimitBloc\] Updating traffic: spent=([\d.]+) GB, max=([\d.]+) GB, limitReached=(\w+)")
APP_ERR = {
    "api_host_lookup": re.compile(rb"Failed host lookup: 'api\.pipster\.win'"),
    "other_host_lookup": re.compile(rb"Failed host lookup: '(?!api\.pipster\.win)"),
    "network_exception": re.compile(rb"Network Exception"),
    "timeout": re.compile(rb"TimeoutException|took longer than"),
    "websocket_fail": re.compile(rb"WebSocketChannelException"),
    "dio_other": re.compile(rb"DioException \[(?!connection)"),
    "cloud_messaging": re.compile(rb"Cloud messaging init timed out"),
}
VPN_ERR = {
    "outbound_fail": re.compile(rb"failed to process outbound traffic"),
    "doh_fail": re.compile(rb"app/dns: failed to retrieve response"),
    "dial_timeout": re.compile(rb"i/o timeout|context deadline exceeded"),
    "conn_refused": re.compile(rb"connection was refused|connection refused|actively refused"),
}
E_LINE = re.compile(rb"\n\[([\w.]+)\]\s+\S+\s+\S+\s+\[E\]\s+")
EV_LINE = re.compile(rb"^\[([\w.]+)\]\s+\S+\s+\S+\s+\[(\w)\]\s+(\[[\w.]+\])?\s*([^\n]{0,40})")
NAME_DATE = re.compile(r"(\d{4}-\d{2}-\d{2})T", re.I)

def kind(name):
    base = name.replace("\\", "/").split("/")[-1].strip().lower()
    if NAME_DATE.search(base) and base.endswith(".log"): return "app_log"
    if "vpn_error" in base: return "vpn_error"
    if "vpn_access" in base: return "vpn_access"
    if "peepster" in base or "logcat" in base: return "logcat"
    return "other"

def norm_event(m):
    comp, lvl, sub, msg = (x.decode("utf-8", "replace") if x else "" for x in m.groups())
    msg = re.sub(r"\d+", "9", msg.split(":")[0]).strip()
    return f"{comp}{sub}:{msg}"[:70]

def parse_zip(args):
    folder, path = args
    r = {"folder": folder, "zip": os.path.basename(path), "headers": [], "traffic": [], "updates": [],
         "app_err": collections.Counter(), "vpn_err": collections.Counter(), "app_files": 0, "app_files_with_err": 0,
         "app_e_lines": 0, "pre_err": collections.Counter(), "dates": []}
    try:
        with zipfile.ZipFile(path) as z:
            for info in z.infolist():
                if info.is_dir(): continue
                k = kind(info.filename)
                if k not in ("app_log", "vpn_error"): continue
                data = z.read(info)
                if k == "app_log":
                    r["app_files"] += 1
                    m = NAME_DATE.search(info.filename)
                    if m: r["dates"].append(m.group(1))
                    h = HDR.search(data, 0, 4000)
                    if h: r["headers"].append((m.group(1) if m else None, *(x.decode() for x in h.groups())))
                    r["traffic"] += [(a.decode(), int(b), int(c)) for a, b, c in TRAF.findall(data)]
                    r["updates"] += [(a.decode(), float(b), float(c), d.decode()) for a, b, c, d in UPD.findall(data)]
                    for name, rx in APP_ERR.items():
                        n = len(rx.findall(data))
                        if n: r["app_err"][name] += n
                    e = E_LINE.search(data)
                    r["app_e_lines"] += data.count(b"]  [E]  ") + data.count(b"] [E] ")
                    if e:
                        r["app_files_with_err"] += 1
                        lines = data[max(0, e.start() - 4000):e.start()].split(b"\n")[-4:]
                        evs = [norm_event(mm) for ln in lines if (mm := EV_LINE.match(ln))]
                        first_err = norm_event(EV_LINE.match(data[e.start() + 1:e.start() + 400]) or EV_LINE.match(b"[x] x x [E] x"))
                        if evs:
                            r["pre_err"][" > ".join(evs[-3:]) + " ⇒ " + first_err] += 1
                else:
                    for name, rx in VPN_ERR.items():
                        n = len(rx.findall(data))
                        if n: r["vpn_err"][name] += n
    except zipfile.BadZipFile:
        r["bad"] = True
    return r

t0 = time.time()
with ProcessPoolExecutor(max_workers=max(2, os.cpu_count() or 2), mp_context=mp.get_context("fork")) as ex:
    RES = list(ex.map(parse_zip, ZIPS, chunksize=4))
print(f"разобрано {len(RES)} архивов за {time.time() - t0:.0f} с")

## 4. H2: реальное потребление трафика на бесплатном тарифе

Сначала выясняем, что означают счётчики: какие бывают значения лимита и сбрасывается ли «использовано» каждые сутки. Затем распределение потребления за день и доля пользователей-дней, упирающихся в лимит.

**Ограничения.** Счётчики есть только у пользователей, приславших логи (787 человек, в основном гости, логи в основном за август–сентябрь 2026). Потребление выше текущего лимита не наблюдается (соединение ограничивается), поэтому верхний хвост распределения занижен (цензурирование).

In [ ]:
def h2_logs():
    tr = pd.DataFrame([(r["folder"], *t) for r in RES for t in r["traffic"]], columns=["user_id", "date", "limit_b", "used_b"])
    up = pd.DataFrame([(r["folder"], *u) for r in RES for u in r["updates"]], columns=["user_id", "date", "spent_gb", "max_gb", "reached"])
    out(f"строк TrafficLimitBloc: {len(tr)} у {tr['user_id'].nunique()} пользователей; строк Updating: {len(up)}")
    if tr.empty: out("счётчиков трафика нет"); return
    tr["limit_mb"], tr["used_mb"] = tr["limit_b"] / 2**20, tr["used_b"] / 2**20
    out("\nчастые значения лимита, МБ (строки / пользователи):")
    lv = tr.groupby(tr["limit_mb"].round(0)).agg(rows=("user_id", "size"), users=("user_id", "nunique")).sort_values("rows", ascending=False)
    out(lv.head(15).to_string())

    # сбрасывается ли «использовано» каждые сутки: сравнить последнее значение дня с первым значением следующего дня
    tr = tr.sort_values(["user_id", "date"]).reset_index(drop=True)
    day = tr.groupby(["user_id", "date"]).agg(first_used=("used_mb", "first"), last_used=("used_mb", "last"),
                                               max_used=("used_mb", "max"), limit_min=("limit_mb", "min"),
                                               limit_max=("limit_mb", "max"), n=("used_mb", "size")).reset_index()
    day["prev_last"] = day.groupby("user_id")["last_used"].shift()
    day["prev_date"] = day.groupby("user_id")["date"].shift()
    nxt = day[pd.to_datetime(day["date"]) - pd.to_datetime(day["prev_date"]) == pd.Timedelta(days=1)]
    out(f"\nпар соседних дней: {len(nxt)}; «использовано» в начале дня меньше, чем в конце прошлого (сброс): "
        f"{(nxt['first_used'] < nxt['prev_last'] - 1).mean():.3f}")

    daily = day[day["limit_max"] < 5 * 1024]           # лимиты до 5 ГБ считаем суточными
    out(f"\nпользователе-дней с суточным лимитом: {len(daily)} у {daily['user_id'].nunique()} пользователей")
    q = daily["max_used"].quantile([.25, .5, .75, .9, .95, .99]).round(1)
    out("потребление за день, МБ (квантили 25/50/75/90/95/99):", q.tolist())
    base = daily["limit_min"]
    out(f"доля дней, где потребление ≥ 80% стартового лимита: {(daily['max_used'] >= 0.8 * base).mean():.3f}; "
        f"≥ 100%: {(daily['max_used'] >= base - 1).mean():.3f}")
    out(f"доля дней, в которые лимит рос (расширение за рекламу): {(daily['limit_max'] > daily['limit_min'] + 1).mean():.3f}")
    ext = daily[daily["limit_max"] > daily["limit_min"] + 1]
    if len(ext):
        out("на сколько вырос лимит за день, МБ (квантили 50/90):", ext.eval("limit_max - limit_min").quantile([.5, .9]).round(0).tolist())

    out("\nчто было бы при другом суточном лимите (доля дней, где наблюдаемое потребление ≥ L; для L выше текущего это нижняя граница):")
    grid = [100, 200, 300, 400, 500, 750, 1000, 1500, 2000]
    sim = pd.Series({L: (daily["max_used"] >= L).mean() for L in grid}).round(4)
    out(sim.to_string())
    user_p90 = daily.groupby("user_id")["max_used"].quantile(0.9)
    out("типичный «тяжёлый» день пользователя (90-й процентиль его дней), МБ, квантили по пользователям 50/75/90:",
        user_p90.quantile([.5, .75, .9]).round(1).tolist())

    if not up.empty:
        out(f"\nUpdating traffic: доля строк с limitReached=true: {(up['reached'] == 'true').mean():.4f}; "
            f"пользователей, хоть раз упиравшихся в лимит: {up.loc[up['reached'] == 'true', 'user_id'].nunique()} из {up['user_id'].nunique()}")
        out("частые значения max, ГБ:", up["max_gb"].round(2).value_counts().head(8).to_dict())

    m = con.sql("SELECT user_id, role, first_regular_paid IS NOT NULL AS paid FROM mart").df()
    daily2 = daily.merge(m, on="user_id", how="left")
    out("\nпотребление за день по роли, МБ (медиана / p90 / доля дней ≥ лимита):")
    out(daily2.groupby("role").apply(lambda g: pd.Series({"days": len(g), "median": g["max_used"].median(),
        "p90": g["max_used"].quantile(.9), "share_hit": (g["max_used"] >= g["limit_min"] - 1).mean()})).round(3).to_string())

    fig, ax = plt.subplots(figsize=(6.4, 3))
    ax.hist(daily["max_used"].clip(upper=1500), bins=np.arange(0, 1525, 25), color=C["blue"], edgecolor="white", linewidth=0.5)
    L0 = float(daily["limit_min"].mode().iloc[0])
    ax.axvline(L0, color=C["ink2"], linestyle="--", linewidth=1)
    ax.annotate(f"текущий лимит {L0:.0f} МБ", xy=(L0, ax.get_ylim()[1] * 0.85), xytext=(L0 + 60, ax.get_ylim()[1] * 0.85),
                color=C["ink2"], va="center", arrowprops=dict(arrowstyle="-", color=C["ink2"], linewidth=0.8))
    ax.set_xlabel("потребление за день, МБ (> 1500 в последнем столбце)"); ax.set_ylabel("пользователе-дней")
    ax.set_title("Суточное потребление на бесплатном тарифе (по клиентским логам)", loc="left")
    fig.tight_layout(); fig.savefig(f"{FIG}/h2_daily_usage.png"); plt.show()
section("H2 Потребление трафика (логи)", h2_logs)

## 5. H4: паттерны, приводящие к ошибкам

Единица анализа — архив (инцидент). Для каждого типа ошибки считаем долю инцидентов, где он встречается, сочетания типов и события непосредственно перед первой ошибкой. Ограничение: логи прислали только пользователи с проблемами, поэтому частоты описывают инциденты, а не всю аудиторию.

In [ ]:
def h4():
    rows = []
    for r in RES:
        hd = sorted([h for h in r["headers"] if h[0]], key=lambda h: h[0])
        last = hd[-1] if hd else (None, None, None, None)
        row = {"folder": r["folder"], "zip": r["zip"], "platform": last[1], "os_version": last[2], "app_version": last[3],
               "last_date": max(r["dates"]) if r["dates"] else None, "app_files": r["app_files"],
               "app_files_with_err": r["app_files_with_err"], "e_lines": r["app_e_lines"]}
        for k in APP_ERR: row[k] = r["app_err"].get(k, 0)
        for k in VPN_ERR: row["vpn_" + k] = r["vpn_err"].get(k, 0)
        rows.append(row)
    global INC
    INC = pd.DataFrame(rows)
    m = con.sql("SELECT user_id AS folder, role, first_regular_paid IS NOT NULL AS ever_paid, platform AS db_platform FROM mart").df()
    INC = INC.merge(m, on="folder", how="left")
    try:
        INC.to_parquet(f"{PARQUET}/log_incidents.parquet")
    except Exception as e:
        out(f"(не удалось сохранить log_incidents.parquet: {e})")
    out(f"инцидентов (архивов): {len(INC)}; пользователей: {INC['folder'].nunique()}; без журнала приложения: {(INC['app_files'] == 0).sum()}")
    cats = list(APP_ERR) + ["vpn_" + k for k in VPN_ERR]
    pres = (INC[cats] > 0)
    out("\nдоля инцидентов с ошибкой данного типа:")
    out(pres.mean().sort_values(ascending=False).round(3).to_string())
    out("\nпо роли пользователя (доля инцидентов):")
    out(pres.groupby(INC["role"]).mean().T.round(3).to_string())
    out("\nчастые сочетания типов ошибок (по инцидентам):")
    combo = pres[[c for c in cats if c not in ("cloud_messaging",)]].apply(lambda r: " + ".join(c for c in r.index if r[c]) or "нет", axis=1)
    out(combo.value_counts().head(12).to_string())
    api = pres["api_host_lookup"]
    out(f"\nинциденты с недоступностью api.pipster.win: {api.mean():.3f}; из них с DoH-сбоями VPN-ядра: "
        f"{pres.loc[api, 'vpn_doh_fail'].mean():.3f}; без них: {pres.loc[~api, 'vpn_doh_fail'].mean():.3f}")

    pre = collections.Counter()
    for r in RES: pre.update(r["pre_err"])
    out("\nсобытия перед первой ошибкой в журнале приложения (3 последних события ⇒ первая ошибка), топ-15:")
    for s, n in pre.most_common(15): out(f"   {n:>6}  {s}")

    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    names = {"api_host_lookup": "API недоступен (DNS)", "network_exception": "сетевое исключение", "timeout": "тайм-аут запроса",
             "websocket_fail": "сбой WebSocket", "other_host_lookup": "другой хост недоступен (DNS)", "dio_other": "прочие ошибки HTTP",
             "vpn_outbound_fail": "VPN: исходящее соединение", "vpn_doh_fail": "VPN: DoH-запрос", "vpn_dial_timeout": "VPN: тайм-аут соединения",
             "vpn_conn_refused": "VPN: соединение отклонено"}
    s = pres[[c for c in names]].mean().rename(index=names).sort_values()
    ax.barh(s.index, s.values, color=[C["orange"] if i.startswith("VPN") else C["blue"] for i in s.index], height=0.6)
    for y, v in enumerate(s.values): ax.text(v + 0.01, y, f"{v:.0%}", va="center", color=C["ink"], fontsize=9)
    ax.set_xlim(0, 1.1); ax.xaxis.set_major_formatter(pct); ax.grid(axis="y", visible=False)
    ax.set_title("Доля инцидентов с ошибкой данного типа", loc="left")
    fig.tight_layout(); fig.savefig(f"{FIG}/h4_error_types.png"); plt.show()
section("H4 Паттерны ошибок (логи)", h4)

## 6. H10: ошибки по версиям приложения и ОС

In [ ]:
def h10():
    inc = INC[INC["app_version"].notna()].copy()
    inc["api_dns"] = inc["api_host_lookup"] > 0
    inc["vpn_any"] = inc[[c for c in inc.columns if c.startswith("vpn_") and c != "vpn_any"]].gt(0).any(axis=1)
    inc["e_per_file"] = inc["e_lines"] / inc["app_files"].clip(lower=1)
    g = inc.groupby(["platform", "app_version"]).agg(incidents=("folder", "size"), users=("folder", "nunique"),
        api_dns=("api_dns", "mean"), vpn_err=("vpn_any", "mean"), e_per_file=("e_per_file", "median"),
        first_seen=("last_date", "min"), last_seen=("last_date", "max"))
    g = g[g["incidents"] >= 10].sort_values("first_seen")
    out("версии с ≥ 10 инцидентами:")
    out(g.round(3).to_string())
    for col, name in [("api_dns", "API недоступен"), ("vpn_any", "ошибки VPN-ядра")]:
        top = inc.groupby("app_version").filter(lambda x: len(x) >= 10)
        if top["app_version"].nunique() >= 2:
            from scipy.stats import chi2_contingency
            ct = pd.crosstab(top["app_version"], top[col])
            if ct.shape[1] == 2:
                chi2, p, *_ = chi2_contingency(ct)
                out(f"различие доли «{name}» между версиями: χ² = {chi2:.1f}, p-value = {p:.2g}")
    inc["os_major"] = inc["os_version"].str.extract(r"^(\d+)")[0]
    o = inc[inc["platform"] == "android"].groupby("os_major").agg(incidents=("folder", "size"), api_dns=("api_dns", "mean"),
                                                                vpn_err=("vpn_any", "mean"))
    out("\nAndroid: по мажорной версии ОС (≥ 10 инцидентов):")
    out(o[o["incidents"] >= 10].round(3).to_string())
    show("mobile_version: даты выхода версий Android и обязательное обновление", """
        SELECT version, CAST(created_at AS DATE) released, json_extract_string(global_settings, '$.requiredUpdate') required_update
        FROM mobile_version WHERE platform = 'android' AND created_at >= '2026-05-01' ORDER BY created_at""")
section("H10 Версии приложения", h10)
out(f"\nГотово. Отчёт: {DRIVE_DIR}/h_logs_results.txt; графики: h2_daily_usage, h4_error_types")
save()